# Notebook 3 — MMR Re-Ranking for Diversity & Fairness (§4.4)
**Implements the Maximal Marginal Relevance re-ranker with learned item embeddings. Demonstrates the '5 Biryanis' problem, λ sensitivity analysis, and the independent restaurant fairness floor.**

> CART-SYNCZ · Team KVK · Zomathon 2025


In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("=" * 60)
print("CART-SYNCZ  |  Notebook 3: MMR Re-Ranking for Diversity")
print("=" * 60)


## ITEM EMBEDDINGS (simulated 8-dim for readability)
## Real system: 64-dim learned embeddings


In [ ]:
# ─────────────────────────────────────────────
# Dim meanings: [spicy, rich, light, sweet, liquid, bread, rice, premium]
ITEM_EMBEDDINGS = {
    "Chicken Biryani":     np.array([0.8, 0.7, 0.1, 0.1, 0.0, 0.0, 0.9, 0.6]),
    "Mutton Biryani":      np.array([0.9, 0.8, 0.1, 0.1, 0.0, 0.0, 0.9, 0.7]),
    "Veg Biryani":         np.array([0.5, 0.5, 0.3, 0.1, 0.0, 0.0, 0.9, 0.4]),
    "Salan":               np.array([0.9, 0.5, 0.2, 0.0, 0.6, 0.0, 0.0, 0.3]),
    "Raita":               np.array([0.0, 0.2, 0.9, 0.1, 0.8, 0.0, 0.0, 0.2]),
    "Sheermal":            np.array([0.1, 0.3, 0.2, 0.5, 0.0, 0.9, 0.0, 0.4]),
    "Pepsi":               np.array([0.0, 0.0, 0.5, 0.5, 1.0, 0.0, 0.0, 0.1]),
    "Lassi":               np.array([0.0, 0.2, 0.6, 0.5, 1.0, 0.0, 0.0, 0.3]),
    "Mango Lassi":         np.array([0.0, 0.2, 0.5, 0.7, 1.0, 0.0, 0.0, 0.4]),
    "Phirni":              np.array([0.0, 0.4, 0.3, 0.9, 0.2, 0.0, 0.0, 0.5]),
    "Gulab Jamun":         np.array([0.0, 0.5, 0.1, 0.9, 0.2, 0.0, 0.0, 0.3]),
    "Kheer":               np.array([0.0, 0.4, 0.3, 0.8, 0.3, 0.0, 0.0, 0.3]),
    "Butter Chicken":      np.array([0.5, 0.9, 0.1, 0.2, 0.3, 0.0, 0.0, 0.7]),
    "Dal Makhani":         np.array([0.3, 0.7, 0.2, 0.1, 0.4, 0.0, 0.0, 0.5]),
    "Naan":                np.array([0.0, 0.2, 0.3, 0.1, 0.0, 0.9, 0.0, 0.2]),
    "Roti":                np.array([0.0, 0.1, 0.4, 0.0, 0.0, 0.8, 0.0, 0.1]),
    "Garlic Naan":         np.array([0.2, 0.3, 0.3, 0.1, 0.0, 0.9, 0.0, 0.3]),
    "Filter Coffee":       np.array([0.1, 0.2, 0.5, 0.3, 1.0, 0.0, 0.0, 0.3]),
    "Masala Chai":         np.array([0.3, 0.2, 0.4, 0.3, 1.0, 0.0, 0.0, 0.2]),
    "Mineral Water":       np.array([0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0]),
}

def cosine_similarity(a, b):
    a_n = a / (np.linalg.norm(a) + 1e-8)
    b_n = b / (np.linalg.norm(b) + 1e-8)
    return float(np.dot(a_n, b_n))


## MMR ALGORITHM  (§4.4)
## Score_MMR = λ × relevance − (1−λ) × max_similarity_to_selected


In [ ]:
# ─────────────────────────────────────────────
def mmr_rerank(candidates_with_scores, embeddings, lam=0.7, k=8):
    """
    candidates_with_scores: list of (item_name, relevance_score) sorted by relevance desc
    embeddings: dict of item_name → np.array
    lam: tradeoff param (higher = more relevance, lower = more diversity)
    k: number of items to return
    """
    selected = []
    remaining = list(candidates_with_scores)

    while len(selected) < k and remaining:
        best_item = None
        best_mmr  = -float("inf")

        for item, rel_score in remaining:
            if not selected:
                # First item — just pick highest relevance
                mmr_score = rel_score
            else:
                sim_to_selected = max(
                    cosine_similarity(embeddings[item], embeddings[s])
                    for s, *_ in selected
                )
                mmr_score = lam * rel_score - (1 - lam) * sim_to_selected

            if mmr_score > best_mmr:
                best_mmr  = mmr_score
                best_item = (item, rel_score, mmr_score)

        selected.append((best_item[0], best_item[1], best_item[2]))
        remaining = [(i, s) for i, s in remaining if i != best_item[0]]

    return selected


## TEST 1: Biryani cart — diverse recommendations


In [ ]:
# ─────────────────────────────────────────────
print("\n" + "─"*60)
print("  TEST 1: Cart = ['Chicken Biryani']  —  Mughlai cart")
print("─"*60)

# DCTN model scored these candidates (already ranked by P(Accept) × AOV)
biryani_candidates = [
    ("Salan",        0.91),
    ("Raita",        0.88),
    ("Phirni",       0.82),
    ("Mutton Biryani", 0.78),  # similar to cart item — should be penalised
    ("Veg Biryani",  0.75),    # similar to cart item — should be penalised
    ("Sheermal",     0.72),
    ("Pepsi",        0.68),
    ("Lassi",        0.65),
    ("Mango Lassi",  0.63),    # very similar to Lassi — should be penalised
    ("Gulab Jamun",  0.60),
    ("Kheer",        0.58),    # similar to Phirni — should be penalised
]

print("\n  Raw DCTN ranked output (before MMR):")
print(f"  {'Rank':<6} {'Item':<22} {'Relevance':>10}")
print("  " + "-"*40)
for i, (item, score) in enumerate(biryani_candidates[:8], 1):
    flag = "  ⚠ near-duplicate" if item in ["Mutton Biryani", "Veg Biryani", "Mango Lassi", "Kheer"] else ""
    print(f"  #{i:<5} {item:<22} {score:>10.2f}{flag}")

print("\n  After MMR Re-ranking (λ=0.7):")
mmr_result = mmr_rerank(biryani_candidates, ITEM_EMBEDDINGS, lam=0.7, k=8)
print(f"  {'Rank':<6} {'Item':<22} {'Relevance':>10} {'MMR Score':>10}  Note")
print("  " + "-"*65)
for i, (item, rel, mmr_s) in enumerate(mmr_result, 1):
    note = ""
    if item in ["Mutton Biryani", "Veg Biryani"]:
        note = "← penalised (too similar to cart)"
    elif item in ["Mango Lassi", "Kheer"]:
        note = "← penalised (too similar to Lassi/Phirni)"
    print(f"  #{i:<5} {item:<22} {rel:>10.2f} {mmr_s:>10.3f}  {note}")


## TEST 2: WITHOUT MMR — the "5 biryanis" problem


In [ ]:
# ─────────────────────────────────────────────
print("\n" + "─"*60)
print("  TEST 2: The '5 Biryanis' problem — why MMR is essential")
print("─"*60)

biryani_variants = [
    ("Chicken Biryani (Half)", 0.89),
    ("Mutton Biryani",         0.87),
    ("Veg Biryani",            0.85),
    ("Egg Biryani",            0.83),
    ("Prawn Biryani",          0.81),
    ("Salan",                  0.78),
    ("Raita",                  0.75),
    ("Lassi",                  0.70),
]

print("\n  Scenario: model scores 5 biryani variants very highly")
print("\n  WITHOUT MMR (naive top-8 by relevance):")
print(f"  {'Rank':<6} {'Item':<28} {'Score':>8}")
print("  " + "-"*44)
for i, (item, score) in enumerate(biryani_variants[:8], 1):
    flag = "  ← USER ANNOYANCE" if "Biryani" in item and i > 1 else ""
    print(f"  #{i:<5} {item:<28} {score:>8.2f}{flag}")

# Add more embeddings for this test
extra_embs = {
    "Chicken Biryani (Half)": np.array([0.8, 0.7, 0.1, 0.1, 0.0, 0.0, 0.9, 0.5]),
    "Egg Biryani":            np.array([0.7, 0.6, 0.1, 0.1, 0.0, 0.0, 0.9, 0.4]),
    "Prawn Biryani":          np.array([0.7, 0.7, 0.1, 0.0, 0.0, 0.0, 0.9, 0.6]),
}
all_embs = {**ITEM_EMBEDDINGS, **extra_embs}

mmr2 = mmr_rerank(biryani_variants, all_embs, lam=0.7, k=6)
print("\n  WITH MMR (λ=0.7):")
print(f"  {'Rank':<6} {'Item':<28} {'Relevance':>10} {'MMR Score':>10}")
print("  " + "-"*55)
for i, (item, rel, mmr_s) in enumerate(mmr2, 1):
    print(f"  #{i:<5} {item:<28} {rel:>10.2f} {mmr_s:>10.3f}")


## TEST 3: Lambda sensitivity analysis


In [ ]:
# ─────────────────────────────────────────────
print("\n" + "─"*60)
print("  TEST 3: Lambda (λ) sensitivity — relevance vs diversity tradeoff")
print("─"*60)
print("""
  λ = 1.0  → Pure relevance ranking (same as raw DCTN output)
  λ = 0.7  → Default: balance relevance with diversity  ← we use this
  λ = 0.5  → Equal weight
  λ = 0.3  → Heavily diversity-focused
""")

for lam in [1.0, 0.7, 0.5, 0.3]:
    result = mmr_rerank(biryani_candidates, ITEM_EMBEDDINGS, lam=lam, k=5)
    items_selected = [r[0] for r in result]
    n_biryanis = sum(1 for i in items_selected if "Biryani" in i)
    print(f"  λ={lam}  →  Top-5: {items_selected}")
    print(f"         Biryani variants in top-5: {n_biryanis}  |  Diverse items: {5-n_biryanis}")

print("\n" + "─"*60)
print("  FAIRNESS FLOOR DEMO (§4.4)")
print("─"*60)

final_recommendations = mmr_rerank(biryani_candidates, ITEM_EMBEDDINGS, lam=0.7, k=8)
selected_items = [r[0] for r in final_recommendations]

# Simulate: no independent restaurant items in top-8
independent_available = "Special House Salan"  # independent restaurant item
independent_emb = np.array([0.85, 0.5, 0.2, 0.0, 0.5, 0.0, 0.0, 0.2])

has_independent = False  # simulating the worst-case scenario

print(f"\n  Top-8 after MMR: {selected_items}")
print(f"  Independent restaurant item present: {has_independent}")
if not has_independent:
    # Insert at position 7 (index 6)
    selected_items.insert(6, independent_available)
    selected_items = selected_items[:8]
    print(f"  → Fairness floor triggered: inserted '{independent_available}' at position 7")
    print(f"  Final top-8: {selected_items}")

print("\n" + "=" * 60)
print("✓  MMR re-ranking demo complete.")
print("   Diversity enforcement ✓  |  λ sensitivity ✓  |  Fairness floor ✓")
print("=" * 60)
